# Banding Debug — 350s

目的：定位原视频无色带、RIFE 已排除时，**第一个出现色带的阶段**。

按生产链路保存同一目标帧：

1. FFmpeg decode
2. BasicVSR++
3. Real-ESRGAN native 4×
4. NPP Lanczos final 2×
5. CPU Lanczos4 对照
6. 编码前 RGB
7. HEVC NVENC 8-bit 解码回读
8. HEVC NVENC 10-bit Main10 解码回读

所有中间图使用 PNG；关键阶段同时保存 NPY。不要用 JPEG。


In [ ]:
!rm -rf /kaggle/working/Real-ESRGAN
!git clone --depth 1 --branch master https://github.com/aksjfds/Real-ESRGAN.git /kaggle/working/Real-ESRGAN
!pip install -q -r /kaggle/working/Real-ESRGAN/requirements.txt
!cd /kaggle/working/Real-ESRGAN && python -m py_compile inference.py inference/*.py encode/*.py inference/models/*.py
!ffmpeg -hide_banner -encoders 2>/dev/null | grep hevc_nvenc || true


In [ ]:
# ===== 参数 =====
from pathlib import Path

REPO_DIR = Path("/kaggle/working/Real-ESRGAN")
INPUT_VIDEO = Path("/kaggle/input/datasets/rustacean1/hanime/ts_2.mp4")

DEBUG_TIME = 350.0

# 必须与产生问题视频时的 START_TIME / TEST_SECONDS 一致，才能复现相同 BVS clip 边界。
PIPELINE_START_TIME = 5 * 60 + 35
PIPELINE_DURATION = 15.0

MODEL = "realesr-animevideov3"
MODEL_PATH = ""
FINAL_SCALE = 2

GPU_ID = 0
BVS_TILE_SIZE = 640
BVS_CLIP_LENGTH = 13
BVS_STRENGTH = 1.0

CQ = 18
NVENC_PRESET = "p7"

# None = 自动找低纹理渐变候选区；也可手工填源分辨率坐标 (x1,y1,x2,y2)
ROI = None

DEBUG_DIR = Path("/kaggle/working/banding_debug_350s")
DEBUG_DIR.mkdir(parents=True, exist_ok=True)
print(DEBUG_DIR)


In [ ]:
import gc
import math
import subprocess
import sys
from types import SimpleNamespace

import cv2
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, str(REPO_DIR))

from inference import runtime_api as base
from inference.clip_source import ClipSource
from inference.basicvsrpp_api import BasicVSRPPConfig
from inference.bvs_runtime import BasicVSRRuntime
from inference.checkpoint_parts import resolve_checkpoint
from inference.sr_runtime import infer_cuda_u8_tensor
from inference.npp_resize import NppLanczosResizer

def save_rgb_png(path, rgb):
    path = Path(path)
    if rgb.dtype not in (np.uint8, np.uint16):
        raise TypeError(rgb.dtype)
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    if not cv2.imwrite(str(path), bgr):
        raise RuntimeError(f"Failed to write {path}")

def save_stage(name, frame, npy=True):
    save_rgb_png(DEBUG_DIR / f"{name}.png", frame)
    if npy:
        np.save(DEBUG_DIR / f"{name}.npy", frame)
    print(f"{name}: shape={frame.shape}, dtype={frame.dtype}")

def to_u8(frame):
    if frame.dtype == np.uint8:
        return frame
    if frame.dtype == np.uint16:
        return np.rint(frame.astype(np.float32) / 257.0).clip(0,255).astype(np.uint8)
    raise TypeError(frame.dtype)

def auto_roi(frame, tile=192):
    # 选一个‘低边缘但仍有亮度渐变’的区域，作为自动色带候选 ROI。
    u8 = to_u8(frame)
    y = cv2.cvtColor(u8, cv2.COLOR_RGB2GRAY).astype(np.float32)
    h, w = y.shape
    best = None
    step = max(64, tile // 2)
    for yy in range(0, max(1, h-tile+1), step):
        for xx in range(0, max(1, w-tile+1), step):
            p = y[yy:min(h,yy+tile), xx:min(w,xx+tile)]
            if p.size < tile*tile*0.7:
                continue
            gx = cv2.Sobel(p, cv2.CV_32F, 1, 0, ksize=3)
            gy = cv2.Sobel(p, cv2.CV_32F, 0, 1, ksize=3)
            edge = float(np.mean(np.hypot(gx,gy)))
            spread = float(np.percentile(p,95)-np.percentile(p,5))
            if spread < 6:
                continue
            score = spread / (1.0 + edge)
            if best is None or score > best[0]:
                best = (score, xx, yy, min(w,xx+tile), min(h,yy+tile))
    return best[1:] if best else (0,0,w,h)

def scaled_roi(roi, src_shape, dst_shape):
    x1,y1,x2,y2 = roi
    sh, sw = src_shape[:2]
    dh, dw = dst_shape[:2]
    return (
        int(round(x1*dw/sw)), int(round(y1*dh/sh)),
        int(round(x2*dw/sw)), int(round(y2*dh/sh)),
    )

def crop(frame, roi):
    x1,y1,x2,y2 = roi
    return frame[y1:y2, x1:x2]

def banding_metrics(frame, roi):
    p = crop(to_u8(frame), roi)
    y = cv2.cvtColor(p, cv2.COLOR_RGB2GRAY).astype(np.int16)
    dx = np.diff(y, axis=1).ravel()
    dy = np.diff(y, axis=0).ravel()
    d = np.concatenate([np.abs(dx), np.abs(dy)])
    nz = d[d>0]
    return {
        "unique_luma": int(np.unique(y).size),
        "luma_range_p95_p05": float(np.percentile(y,95)-np.percentile(y,5)),
        "plateau_ratio": float(np.mean(d==0)),
        "step1_ratio_nonzero": float(np.mean(nz==1)) if nz.size else 0.0,
        "small_step_ratio_nonzero": float(np.mean(nz<=2)) if nz.size else 0.0,
    }

def decode_first_rgb48(video_path, width, height):
    cmd = [
        "ffmpeg","-hide_banner","-loglevel","error","-i",str(video_path),
        "-frames:v","1","-f","rawvideo","-pix_fmt","rgb48le","pipe:1",
    ]
    raw = subprocess.check_output(cmd)
    expected = width*height*3*2
    if len(raw) != expected:
        raise RuntimeError(f"decoded bytes {len(raw)} != {expected}")
    return np.frombuffer(raw, dtype="<u2").reshape(height,width,3).copy()


In [ ]:
# ===== 1. 生产 decode + ClipSource 定位 350s 对应真实 BVS clip =====
info = base.probe_video(INPUT_VIDEO, "ffprobe")
fps = info.fps
source_rate_text = f"{info.fps_num}/{info.fps_den}"

if not (PIPELINE_START_TIME <= DEBUG_TIME <= PIPELINE_START_TIME + PIPELINE_DURATION + 1e-9):
    raise ValueError("DEBUG_TIME 必须位于 PIPELINE_START_TIME / PIPELINE_DURATION 范围内")

# ‘第350秒’取 350.000s 之前最后一张源帧；24fps 时约为 349.958333s。
target_frame_id = max(0, int(math.floor((DEBUG_TIME - PIPELINE_START_TIME) * fps - 1e-9)))
target_timestamp = PIPELINE_START_TIME + target_frame_id / fps

reader = base.RawVideoReader(
    INPUT_VIDEO, "ffmpeg",
    info.width, info.height,
    source_rate_text,
    float(PIPELINE_START_TIME),
    float(PIPELINE_DURATION),
    info.bit_depth,
)
clip_source = ClipSource(
    reader,
    clip_length=BVS_CLIP_LENGTH,
    overlap=2,
    scene_threshold=0.30,
)

next_emitted_id = 0
target_clip = None
target_pos = None
target_emit_range = None

while True:
    task = clip_source.next_task()
    if task is None:
        break
    frames, emit_start, emit_end = task
    emitted_count = emit_end - emit_start
    if next_emitted_id <= target_frame_id < next_emitted_id + emitted_count:
        target_clip = frames
        target_pos = emit_start + (target_frame_id - next_emitted_id)
        target_emit_range = (emit_start, emit_end)
        break
    next_emitted_id += emitted_count

clip_source.close()

if target_clip is None:
    raise RuntimeError(
        f"没有找到目标帧 id={target_frame_id}; "
        "请确认 PIPELINE_START_TIME / PIPELINE_DURATION 与产生问题视频时一致"
    )

decode_frame = np.ascontiguousarray(target_clip[target_pos])
save_stage("00_decode", decode_frame)

print(
    f"source={info.width}x{info.height} {fps:.6f}fps {info.pix_fmt} {info.bit_depth}-bit\n"
    f"requested DEBUG_TIME={DEBUG_TIME:.6f}s\n"
    f"actual source frame={target_frame_id}, timestamp={target_timestamp:.6f}s\n"
    f"BVS clip frames={len(target_clip)}, target_pos={target_pos}, emit={target_emit_range}"
)

if ROI is None:
    ROI = auto_roi(decode_frame)
print("ROI(source coordinates) =", ROI)


In [ ]:
# ===== 2. BasicVSR++：完全复用生产 runtime =====
device = torch.device(f"cuda:{GPU_ID}")
torch.cuda.set_device(GPU_ID)

checkpoint = resolve_checkpoint(REPO_DIR / "inference" / "weights")
bvs = BasicVSRRuntime(
    BasicVSRPPConfig(
        gpu_id=GPU_ID,
        strength=BVS_STRENGTH,
        clip_length=BVS_CLIP_LENGTH,
        clip_overlap=2,
        tile_size=BVS_TILE_SIZE,
        tile_pad=32,
        fp16=True,
        scene_threshold=0.30,
        model_path=str(checkpoint),
    ),
    checkpoint_dir=REPO_DIR / "inference" / "weights",
)

bvs_frames = bvs.enhance_clip(target_clip)
bvs_frame = np.ascontiguousarray(bvs_frames[target_pos])
save_stage("01_basicvsrpp", bvs_frame)

bvs.close()
del bvs, bvs_frames
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# ===== 3. Real-ESRGAN native 4x → NPP Lanczos final 2x =====
args = SimpleNamespace(model=MODEL, model_path=MODEL_PATH)
model_paths = base.resolve_model_paths(args)
worker_cfg = base.WorkerConfig(MODEL, model_paths)
sr_model, native_scale = base.load_worker_model(worker_cfg, device)

sr_cuda = infer_cuda_u8_tensor(sr_model, bvs_frame, device)
torch.cuda.synchronize()
sr_native = sr_cuda.cpu().numpy()
save_stage("02_realesrgan_native4x", sr_native)

out_w = int(round(info.width * FINAL_SCALE))
out_h = int(round(info.height * FINAL_SCALE))

resizer = NppLanczosResizer(device)
npp_cuda = resizer.resize_batch(sr_cuda.unsqueeze(0), out_h, out_w)[0]
torch.cuda.synchronize()
npp_frame = npp_cuda.cpu().numpy()
save_stage("03_npp_lanczos_final2x", npp_frame)

cpu_lanczos = cv2.resize(sr_native, (out_w, out_h), interpolation=cv2.INTER_LANCZOS4)
save_stage("03b_cpu_lanczos4_final2x", np.ascontiguousarray(cpu_lanczos))

preencode = np.ascontiguousarray(npp_frame)
save_stage("04_preencode_rgb8", preencode, npy=False)

del sr_model, sr_cuda, npp_cuda
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# ===== 4. HEVC NVENC 8-bit / 10-bit Main10 对照 =====
def encode_one_frame(path, frame, ten_bit):
    h, w = frame.shape[:2]
    cmd = ["ffmpeg","-y","-hide_banner","-loglevel","error","-f","rawvideo"]
    if ten_bit:
        raw = frame.astype(np.uint16) * 257
        cmd += ["-pix_fmt","rgb48le"]
    else:
        raw = frame
        cmd += ["-pix_fmt","rgb24"]

    cmd += [
        "-s:v",f"{w}x{h}","-r","24","-i","pipe:0",
        "-frames:v","1","-an",
        "-c:v","hevc_nvenc",
        "-gpu",str(GPU_ID),
        "-preset",NVENC_PRESET,
        "-tune","hq",
        "-rc","vbr",
        "-cq",str(CQ),
        "-b:v","0",
        "-multipass","fullres",
        "-spatial_aq","1",
        "-temporal_aq","1",
        "-rc-lookahead","32",
        "-bf","3",
    ]
    if ten_bit:
        cmd += ["-profile:v","main10","-pix_fmt","p010le"]
    else:
        cmd += ["-pix_fmt","yuv420p"]
    cmd += ["-tag:v","hvc1",str(path)]

    result = subprocess.run(
        cmd, input=np.ascontiguousarray(raw).tobytes(),
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr.decode(errors="replace"))

enc8 = DEBUG_DIR / "05_nvenc_8bit.mp4"
enc10 = DEBUG_DIR / "06_nvenc_10bit_main10.mp4"

encode_one_frame(enc8, preencode, ten_bit=False)
encode_one_frame(enc10, preencode, ten_bit=True)

dec8_16 = decode_first_rgb48(enc8, out_w, out_h)
dec10_16 = decode_first_rgb48(enc10, out_w, out_h)
save_stage("05_nvenc_8bit_decoded16", dec8_16)
save_stage("06_nvenc_10bit_decoded16", dec10_16)

print("8-bit:", enc8)
print("10-bit:", enc10)


In [ ]:
# ===== 5. 自动指标 + ROI 可视化 =====
stages = {
    "decode": decode_frame,
    "basicvsrpp": bvs_frame,
    "realesrgan_native4x": sr_native,
    "npp_final2x": npp_frame,
    "cpu_lanczos_final2x": cpu_lanczos,
    "preencode_rgb8": preencode,
    "nvenc_8bit_decoded": dec8_16,
    "nvenc_10bit_decoded": dec10_16,
}

rows = []
for name, frame in stages.items():
    r = scaled_roi(ROI, decode_frame.shape, frame.shape)
    m = banding_metrics(frame, r)
    m.update({"stage":name, "roi":str(r), "dtype":str(frame.dtype)})
    rows.append(m)

metrics = pd.DataFrame(rows).set_index("stage")
display(metrics)
metrics.to_csv(DEBUG_DIR / "banding_metrics.csv")

fig, axes = plt.subplots(2, 4, figsize=(22, 11))
for ax, (name, frame) in zip(axes.ravel(), stages.items()):
    r = scaled_roi(ROI, decode_frame.shape, frame.shape)
    patch = crop(to_u8(frame), r)
    ax.imshow(patch, interpolation="nearest")
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.savefig(DEBUG_DIR / "banding_roi_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

print("\n诊断规则：")
print("1) decode 正常、BasicVSR++ 首次出现色带 → BVS/其 uint8 量化边界。")
print("2) BVS 正常、Real-ESRGAN native4x 首次出现 → SR 或 SR uint8 量化。")
print("3) SR native4x 正常、NPP 2x 出现；CPU Lanczos 正常 → NPP 实现。")
print("4) NPP 与 CPU Lanczos 都出现 → Lanczos 下采样/uint8 边界本身。")
print("5) preencode 正常、NVENC 8-bit 出现而 Main10 明显改善 → yuv420p/8-bit 编码量化。")
print("6) preencode 已有色带 → 编码器不是首因。")
print("\n输出目录:", DEBUG_DIR)
print("重点看:", DEBUG_DIR / "banding_roi_comparison.png")
